# FASTA sequence curation and metadata annotation

**Created by Kate Otter.**

This notebook combines FASTA records, species information, accession numbers, and paper DOIs into sequences and metadata ready for alignment and tree building. Local sequence labels can be retained when no public accession is available.

### Run order

1. Edit and run **Imports & Global Configuration**. Check the input FASTA, source DOI CSV, output folder, and NCBI email.
2. Run the helper-definition cells down through **SignalP 6 trimming**. These define the functions used below.
3. Run **Preparation** to create the metadata review CSV. Open it, fill unresolved entries, and save it before continuing.
4. Run **Pipeline Execution (main)**. It rereads the saved review sheet and may ask you to choose duplicate records or supply missing species.
5. Run optional analysis sections only when needed, after checking their paths and switches.

The main pipeline does not overwrite the input FASTA. All files created on the same date from runs will be overwritten if ran on the same day.

### Default output files
| File | Purpose |
| --- | --- |
| `metadata_review.csv` | Editable species, gene, accession, and DOI information; used as input to the main pipeline. |
| `alignment_ready.csv` | Metadata for the final alignment FASTA, including its exact headers. |
| `alignment_ready.fasta` | Sequences with consistent, unique alignment labels. |
| `deduplicated_original_headers.fasta` | Deduplicated sequences retaining the original FASTA headers, before any SignalP trimming. |

Output names include the dataset and run date. Lookup caches are JSON files used to avoid repeating online requests; they are not review sheets.

### How sequence identifiers are handled

Each input record receives a unique internal `local_id`. Repeated labels receive suffixes such as `__record2`; `original_header` preserves the input header. Keep `local_id` unchanged in the review CSV so edits can be matched back to the correct record.

A local label, a genomic model ID, and a public protein accession are different identifiers. The preparation step can search recognized model IDs and assign a protein accession when a candidate sequence matches exactly. Unresolved records remain available for manual annotation and can still be included in the final FASTA.

## Imports & Global Configuration — run first

Set paths and options here before running the helpers. Imports require Biopython, pandas, and requests. Use the Python environment attached to this notebook's kernel.

| Setting | Effect |
| --- | --- |
| `INPUT_FASTA`, `DATASET_ID`, `BASE_OUTPUT_DIR` | Select the source sequences and output location. |
| `PAPER_ACCESSIONS_CSV` | Input accession-to-DOI sheet; verify that the file exists. |
| `LOCAL_METADATA_CSV` | Completed review sheet read by the pipeline; defaults to this run's review CSV. |
| `PREP_RUN_LOOKUPS` | Enable remote metadata searches during preparation. |
| `ANNOTATE_TAXONOMY` | Retrieve taxonomic ranks from NCBI during the main pipeline. |
| `PROMPT_FOR_SPECIES`, `INTERACTIVE_DEDUP` | Enable interactive species entry and duplicate selection. |
| `ALIGNMENT_IDENTIFIER`, `HEADER_TAXON_RANK` | Choose identifiers and taxonomy used in alignment headers. |
| `CREATE_UIDS` | Enable additional UID-based output. |
| `WRITE_DOI_REPORT` | Write the optional DOI matching audit CSV. |
| `EXPORT_SIGNALP_INPUT`, `REMOVE_SIGNAL_PEPTIDES` | Prepare external SignalP input or apply supplied predictions. |
| `RUN_INTERPRO`, `RUN_FASTA_RESYNC`, `RUN_PEPTIDE_ANALYSIS` | Enable the optional sections at the end. |

`MIN_LENGTH` takes effect only when the length-filter call is enabled in the main pipeline. `PREP_RUN_LOOKUPS=False` does not disable main-pipeline taxonomy or DOI requests. Changing the run date changes the default review-sheet path; point `LOCAL_METADATA_CSV` to an earlier completed sheet if you want to reuse it.

`REVIEW_COLUMNS` and `FINAL_COLUMNS` control exported CSV columns. Other working columns remain available in memory.

In [ ]:
from Bio import SeqIO, Entrez
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import pandas as pd
import re
import time
from urllib.error import HTTPError
from datetime import datetime
from pathlib import Path

Entrez.email = "kxo233@case.edu"

# ===== USER CONFIG =====
INPUT_FASTA = Path(r"G:\My Drive\Sondhi_Lab_Otter_Data\Projects\Opsins\bioinformatics\fasta\20261009\opsins_20261009_deduplicated_original_headers.fasta")

INTERACTIVE_DEDUP = True
CREATE_UIDS = False  # Set True to assign dataset-specific UIDs

MIN_LENGTH = 10 #60 #300
ALIGNMENT_IDENTIFIER = "accession"  # NCBI or UniProt identifier, or set "uid"
PROMPT_FOR_UNIPROT_ID = False # Optionally ask for a UniProt ID when accession is blank
REMOVE_SIGNAL_PEPTIDES = False  # Trim only sequences SignalP predicts have a signal peptide
SIGNALP_RESULTS = None  # Path to SignalP 6 prediction_results.txt when trimming
HEADER_SUFFIX = None  # Optional text appended after the accession, e.g. "identifier"
HEADER_TAXON_RANK = "order"  # None, "family", or "order"; appended after HEADER_SUFFIX

DATASET_ID = "opsins"
RUN_DATE = datetime.today().strftime("%Y%m%d")  # e.g. 20260114

TAXONOMIC_RANKS = [
    "phylum", "class", "order", "family"
    # "superphylum", "phylum", "class", "subclass", "order", "family"
    #  "superphylum", "phylum", "class", "order", "family"
]

ADD_SOURCE_DOIS = True
# CSV columns: paper_accession (or accession), source_doi. One pair per row.
PAPER_ACCESSIONS_CSV = Path(r"G:\My Drive\Sondhi_Lab_Otter_Data\Projects\Opsins\bioinformatics\fasta\opsins_accession_source_proteins.csv")
# Persistent cache reused across runs and datasets.
DOI_LOOKUP_CACHE = Path("doi_protein_lookup_cache.json")

BASE_OUTPUT_DIR = Path(r"G:\My Drive\Sondhi_Lab_Otter_Data\Projects\Opsins\bioinformatics\fasta")
OUTPUT_DIR = BASE_OUTPUT_DIR / RUN_DATE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ==== INTERPRO SCAN CONFIG ====
INTERPRO_BATCH_SIZE = 999
INTERPRO_DIR = OUTPUT_DIR / "interpro"
INTERPRO_DIR.mkdir(exist_ok=True)


# Local metadata: Preparation writes the review CSV used below.
LOCAL_METADATA_CSV = None  # Reassigned to REVIEW_METADATA_CSV below.
ANNOTATE_TAXONOMY = True  # False for offline runs
PROMPT_FOR_SPECIES = True  # Optionally ask for species when taxon is ambiguous

# Preparation is run separately, before the main pipeline.
PREP_RUN_LOOKUPS = True  # False for offline review sheets
PREP_MAX_CANDIDATES = 20  # Ambiguous/broad searches are left for review
PREP_LOOKUP_CACHE = OUTPUT_DIR / "local_model_lookup_cache.json"
REVIEW_METADATA_CSV = OUTPUT_DIR / f"{DATASET_ID}_{RUN_DATE}_metadata_review.csv"
SPECIES_REVIEW_CSV = OUTPUT_DIR / f"{DATASET_ID}_{RUN_DATE}_species_to_fill.csv"
LOCAL_METADATA_CSV = REVIEW_METADATA_CSV  # Main pipeline imports completed review sheet


# Output controls: the default run creates two CSVs and two FASTAs.
WRITE_DOI_REPORT = False
EXPORT_SIGNALP_INPUT = False  # Enable to prepare input for an external SignalP run.
RUN_INTERPRO = False
RUN_FASTA_RESYNC = False
RUN_PEPTIDE_ANALYSIS = False
REVIEW_COLUMNS = [
    "local_id", "original_header", "species", "gene", "accession",
    "identifier_source", "source_doi", "mapping_status", "missing_fields",
]
FINAL_COLUMNS = [
    "alignment_header", "local_id", "species", "gene", "accession",
    "identifier_source", "source_doi",
]
FINAL_COLUMNS += TAXONOMIC_RANKS if ANNOTATE_TAXONOMY else []
# Add column names above if needed. Full data stays available in memory.

def export_columns(frame, columns):
    return frame.reindex(columns=list(dict.fromkeys(columns)), fill_value="")

def write_doi_report(audit):
    if WRITE_DOI_REPORT:
        audit.to_csv(out("doi_matches_for_review.csv"), index=False)


## Pipeline logging — helper definitions

Record stage names, record counts, and output paths in memory. The compact-output workflow does not write an additional pipeline-log CSV.

In [ ]:
PIPELINE_LOG = []

def log_stage(stage, df=None, outputs=None, notes=""):
    entry = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "dataset_id": DATASET_ID,
        "run_date": RUN_DATE,
        "stage": stage,
        "n_sequences": len(df) if df is not None else "",
        "outputs": "; ".join(str(o) for o in outputs) if outputs else "",
        "notes": notes,
    }
    PIPELINE_LOG.append(entry)

def write_pipeline_log():
    log_df = pd.DataFrame(PIPELINE_LOG)
    # Keep the log in memory; no additional CSV.
    return log_df

## Accession and species extraction — helper definitions

Recognize common NCBI and UniProt identifiers and species names in supported FASTA header formats. Abbreviations and arbitrary local labels may need manual annotation. Automatic gene-name extraction here is tailored to opsin labels; fill other gene names in the review sheet.

In [ ]:
ACCESSION_RE = re.compile(
    r"(?<![A-Za-z0-9])((?:[NXYW][MP]_\d+|[A-Z]{4}\d{8,}|[A-Z]{2}\d{6}|[A-Z]{3}\d{5,})(?:\.\d+)?)(?![A-Za-z0-9])"
)


def extract_accession(text):
    # Keep NCBI/GenBank accessions separate from UniProt identifiers.
    m = ACCESSION_RE.search(text)
    return m.group(1) if m else ""


UNIPROT_ACCESSION_RE = re.compile(
    r"(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})"
)


def extract_uniprot_identifier(text):
    # Typical UniProt FASTA headers start sp|P12345|ENTRY_NAME or tr|A0A...|ENTRY_NAME.
    match = re.search(r"(?:^|[\s_])(?:sp|tr)\|([^|\s]+)\|", str(text), re.I)
    if match:
        return match.group(1)
    # Also accept an accession as the first token in a bare UniProt FASTA header.
    token = str(text).split(None, 1)[0] if str(text).strip() else ""
    return token if UNIPROT_ACCESSION_RE.fullmatch(token) else ""


def prompt_for_missing_uniprot_ids(df):
    df = df.copy()
    for i, row in df.iterrows():
        if str(row["accession"]).strip():
            continue
        print("\nNo NCBI or UniProt identifier for:", row["original_header"])
        value = input("Enter a UniProt identifier (or Enter to leave blank): ").strip()
        if value:
            df.at[i, "accession"] = value
            df.at[i, "identifier_source"] = "UniProt"
    return df

def extract_species(header):
    """
    Extract a binomial species name from square brackets in a FASTA header.

    Example:
    >XP_394233.5 protein name [Apis mellifera]
    returns:
    Apis mellifera
    """
    if pd.isna(header):
        return ""

    header = str(header).strip()

    # Find the final bracketed taxon name in the header.
    matches = re.findall(r"\[([^\[\]]+)\]", header)

    if not matches:
        os_match = re.search(r"\bOS=([A-Z][a-z]+) ([a-z][a-z.-]+)", header)
        if os_match:
            return " ".join(os_match.groups())
        name = re.match(r"^([A-Z][a-z]+)_([a-z][a-z.-]+)(?:_|$)", header)
        return " ".join(name.groups()) if name else ""

    taxon = matches[-1].strip()

    # Retain the first two words: Genus species.
    words = taxon.split()

    if len(words) < 2:
        return ""

    genus = words[0]
    species = words[1]

    # Confirm that it resembles a binomial species name.
    if not re.fullmatch(r"[A-Z][A-Za-z.-]*", genus):
        return ""

    if not re.fullmatch(r"[a-z][A-Za-z.-]*", species):
        return ""

    return f"{genus} {species}"


def out(name):
    return OUTPUT_DIR / f"{DATASET_ID}_{RUN_DATE}_{name}"


# FILTERING THE FASTA

def filter_by_min_length(df, min_len):
    return df[df["sequence"].str.len() >= min_len].copy()


def filter_by_cc_motif(df):
    return df[df["motif"] != "NA"].copy()


def filter_by_accession_list(df, accession_set):
    return df[df["accession"].isin(accession_set)].copy()


## FASTA ingestion and local metadata — helper definitions

Read sequences, convert them to uppercase, preserve original headers, and assign unique local IDs. Import nonblank review-sheet values by `local_id`. Keep one review row per local ID; extra columns do not affect the import.

In [ ]:
def ingest_fasta(fasta_path):
    rows = []
    uid_counter = 1
    records = list(SeqIO.parse(fasta_path, "fasta"))
    labels = [rec.id.split("|")[0] for rec in records]
    reserved = set(labels)
    used = set()
    counts = {}

    for rec, label in zip(records, labels):
        counts[label] = counts.get(label, 0) + 1
        local_id = label
        if local_id in used:
            suffix = counts[label]
            local_id = f"{label}__record{suffix}"
            while local_id in reserved or local_id in used:
                suffix += 1
                local_id = f"{label}__record{suffix}"
        used.add(local_id)
        ncbi = extract_accession(rec.description)
        uniprot = extract_uniprot_identifier(rec.description) if not ncbi else ""
        row = {
            "sequence": str(rec.seq).upper(),
            "local_id": local_id,
            "original_local_id": label,
            "gene": (re.search(r"(?i)(?:^|[_|])(Rh[1-7]|Peropsin|C-opsin)(?=[_|\s]|$)", rec.description).group(1)
                     if re.search(r"(?i)(?:^|[_|])(Rh[1-7]|Peropsin|C-opsin)(?=[_|\s]|$)", rec.description) else ""),
            "source_doi": "",
            "original_header": rec.description,
            "accession": ncbi or uniprot,  # Blank when neither is found.
            "identifier_source": "NCBI" if ncbi else ("UniProt" if uniprot else ""),
            "species": extract_species(rec.description),
        }
        if CREATE_UIDS:
            row["uid"] = f"{DATASET_ID}{uid_counter:06d}"
            uid_counter += 1
        rows.append(row)

    return pd.DataFrame(rows, columns=["sequence", "local_id", "original_local_id", "gene", "source_doi", "original_header", "accession", "identifier_source", "species"] + (["uid"] if CREATE_UIDS else []))


def apply_local_metadata(df, metadata_path=None):
    """Join by exact local_id. Blank entries never erase existing metadata."""
    df = df.copy()
    columns = ["local_id", "species", "gene", "source_doi", "accession", "identifier_source", "mapping_status", "model_id", "mapping_note"]
    for column in columns:
        if column not in df:
            df[column] = ""
    if metadata_path is None:
        return df
    if not Path(metadata_path).is_file():
        print(f"Metadata file not found: {metadata_path}; continuing without it.")
        return df
    mapping = pd.read_csv(metadata_path, dtype=str, keep_default_na=False)
    if "local_id" not in mapping:
        raise ValueError("Metadata CSV requires a local_id column.")
    if mapping["local_id"].duplicated().any():
        raise ValueError("Metadata CSV has duplicate local_id values.")
    mapping = mapping.set_index("local_id")
    for column in columns[1:]:
        if column not in mapping:
            continue
        values = df["local_id"].map(mapping[column]).fillna("").str.strip()
        mask = values.ne("")
        df.loc[mask, column] = values[mask]
    unmatched = mapping.index.difference(df["local_id"])
    if len(unmatched):
        print(f"Warning: {len(unmatched)} metadata IDs are not in this FASTA.")
    return df


## Sequence deduplication — helper definition

Group records by identical sequence and retain one record per sequence. This applies across species as well as within a species. With interactive selection enabled, choose which header and metadata to retain for duplicate groups; otherwise the first record is retained.

In [ ]:
def deduplicate(df, interactive=True):
    keep_rows = []

    for seq, group in df.groupby("sequence"):
        if len(group) == 1:
            keep_rows.append(group.iloc[0])
            continue

        headers = group["original_header"].unique()
        if len(headers) == 1:
            keep_rows.append(group.iloc[0])
            continue

        if interactive:
            print("\nDuplicate sequence with different headers:")
            for i, row in enumerate(group.itertuples()):
                print(f"{i+1}: {row.original_header}")
            choice = int(input("Choose entry to keep: ")) - 1
            keep_rows.append(group.iloc[choice])
        else:
            keep_rows.append(group.iloc[0])

    return pd.DataFrame(keep_rows).reset_index(drop=True)



## Species resolution and taxonomy — helper definitions

Prompt for species still missing after deduplication when enabled; pressing Enter leaves an entry unresolved. Optional NCBI searches retrieve the configured taxonomic ranks. Check ambiguous species names in the review sheet before relying on their taxonomy.

In [ ]:
def resolve_unknown_species(df):
    unknowns = df[df["species"] == ""]

    if unknowns.empty:
        print("No unknown species detected.")
        return df

    print(f"\n{len(unknowns)} sequences with unknown species.")
    print("You will be prompted to assign species names.\n")

    for i, row in unknowns.iterrows():
        print("\nHeader:")
        print(row["original_header"])
        species = input("Enter species (Genus species) or press Enter to skip: ").strip()
        if species:
            df.loc[i, "species"] = species

    return df


def fetch_taxonomy(species, ranks):
    handle = Entrez.esearch(db="taxonomy", term=species)
    result = Entrez.read(handle)
    handle.close()

    if not result["IdList"]:
        return {r: "" for r in ranks}

    taxid = result["IdList"][0]
    try:
        with Entrez.efetch(db="taxonomy", id=taxid) as handle:
            record = Entrez.read(handle)[0]
    except HTTPError as exc:
        if exc.code != 400:
            raise
        print(f"Taxonomy fetch failed for {species!r} (taxid {taxid}): HTTP 400; leaving taxonomy blank")
        return {r: "" for r in ranks}

    lineage = record.get("LineageEx", [])
    out = {r: "" for r in ranks}

    for node in lineage:
        r = node["Rank"].lower()
        if r in out:
            out[r] = node["ScientificName"]

    return out

def annotate_taxonomy(df, ranks, sleep=0.3):
    df = df.copy()
    for rank in ranks:
        if rank not in df.columns:
            df[rank] = ""
    cache = {}
    for i, row in df.iterrows():
        sp = row["species"]
        if not sp:
            continue
        if sp not in cache:
            cache[sp] = fetch_taxonomy(sp, ranks)
            time.sleep(sleep)
        for r in ranks:
            df.loc[i, r] = cache[sp].get(r, "")
    return df



## Source DOI matching — helper definitions

Read the supplied source DOI CSV rather than asking you to retype DOI information. Use `source_doi` with an identifier column such as `paper_accession` or `accession`; local/model identifiers are also supported. A missing input file raises an error so a wrong path is visible.

Accession comparisons ignore terminal version suffixes such as `.1` and `.2`. Remote matching can also follow nucleotide records to their CDS protein accessions. Existing DOI values are retained and multiple matches are combined. DOI matching attributes a source paper; it does not verify that two sequences are identical or replace the sequence's accession.

In [ ]:
import json

def _doi_clean(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    return "" if text.lower() in {"", "na", "n/a", "nan", "none"} else text

def _doi_root(value):
    return re.sub(r"\.\d+$", "", value)

def _doi_proteins(accession, cache, cache_path):
    if accession in cache:
        return cache[accession]
    if not Entrez.email:
        raise ValueError("Set Entrez.email in the configuration cell.")
    with Entrez.efetch(db="nuccore", id=accession, rettype="gb", retmode="text") as handle:
        records = list(SeqIO.parse(handle, "genbank"))
    records = [r for r in records if r.id == accession or (
        not re.search(r"\.\d+$", accession) and _doi_root(r.id) == accession)]
    if not records:
        raise ValueError("Requested accession/version not returned by NCBI.")
    proteins = sorted({pid for r in records for f in r.features
                       if f.type == "CDS" for pid in f.qualifiers.get("protein_id", [])})
    cache[accession] = proteins
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache_path.with_suffix(cache_path.suffix + ".tmp")
    tmp.write_text(json.dumps(cache, indent=2), encoding="utf-8")
    tmp.replace(cache_path)
    return proteins

def annotate_source_dois(df, paper_csv, accession_column="accession",
                         cache_file="doi_protein_lookup_cache.json", allow_remote=True):
    """Assign DOIs by accession root, ignoring numeric version suffixes."""
    if paper_csv is None:
        raise ValueError("Set PAPER_ACCESSIONS_CSV to your DOI input CSV.")

    paper_csv = Path(paper_csv)

    if not paper_csv.is_file():
        raise FileNotFoundError(
            f"Cannot find the DOI input CSV:\n{paper_csv}\n"
            "Correct PAPER_ACCESSIONS_CSV in the configuration cell."
        )

    print(f"Reading DOI input CSV: {paper_csv.resolve()}")
    papers = pd.read_csv(paper_csv, dtype=str, keep_default_na=False)
    # Accept the accession column used in previously generated DOI tables too.
    if "paper_accession" not in papers:
        for alias in ["accession", "local_id", "model_id"]:
            if alias in papers:
                papers = papers.rename(columns={alias: "paper_accession"})
                break
    if not {"paper_accession", "source_doi"}.issubset(papers.columns):
        raise ValueError("Paper CSV needs paper_accession, accession, local_id, or model_id, plus source_doi.")
    result = df.copy()
    if accession_column not in result:
        result[accession_column] = ""
    accessions = [_doi_clean(v) for v in result[accession_column]]
    exact, roots = {}, {}
    for pos, acc in enumerate(accessions):
        if acc:
            exact.setdefault(acc, []).append(pos)
            roots.setdefault(_doi_root(acc), []).append(pos)
    # Match local identifiers and genomic models from the existing DOI sheet too.
    local_matches = {}
    for pos, (_, record) in enumerate(result.iterrows()):
        keys = [_doi_clean(record.get("local_id", "")), _doi_clean(record.get("model_id", "")),
                local_model_id(record.get("original_header", ""))]
        for key in set(keys) - {""}:
            local_matches.setdefault(key, []).append(pos)
    old = result["source_doi"] if "source_doi" in result else [""] * len(result)
    dois = [list(dict.fromkeys(x.strip() for x in _doi_clean(v).split(";") if x.strip())) for v in old]
    cache_path = Path(cache_file)
    cache = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else {}
    columns = ["paper_accession", "source_doi", "candidate_protein_id",
               "metadata_accession", "status", "detail"]
    audit = []
    def report(acc, doi, candidate="", matched="", status="", detail=""):
        audit.append(dict(zip(columns, [acc, doi, candidate, matched, status, detail])))
    for _, row in papers.drop_duplicates().iterrows():
        acc = _doi_clean(row["paper_accession"])
        doi = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", _doi_clean(row["source_doi"]), flags=re.I)
        if not acc or not doi:
            report(acc, doi, status="missing_input")
            continue
        local_positions = local_matches.get(acc, []) if acc not in exact else []
        if local_positions:
            for pos in local_positions:
                if doi not in dois[pos]:
                    dois[pos].append(doi)
                report(acc, doi, matched=accessions[pos], status="local_identifier_match",
                       detail=f"Matched local record {result.iloc[pos].get("local_id", "")}")
            continue
        is_protein = bool(re.fullmatch(r"(?:(?:NP|XP|YP|WP|AP|ZP)_\d+|[A-Z]{3}\d+)(?:\.\d+)?", acc))
        if _doi_root(acc) in roots or is_protein:
            candidates, method = [acc], "exact_accession"
        else:
            if not allow_remote:
                report(acc, doi, status="remote_lookup_skipped", detail="Offline preparation; nucleotide-to-protein lookup deferred.")
                continue
            try:
                candidates = _doi_proteins(_doi_root(acc), cache, cache_path)
                method = "nucleotide_to_protein"
            except Exception as error:
                report(acc, doi, status="lookup_failed", detail=str(error))
                continue
            if not candidates:
                report(acc, doi, status="no_annotated_protein")
                continue
        for candidate in candidates:
            positions = roots.get(_doi_root(candidate), [])
            if positions:
                for pos in positions:
                    if doi not in dois[pos]:
                        dois[pos].append(doi)
                    version_differs = candidate != accessions[pos]
                    report(
                        acc, doi, candidate, accessions[pos],
                        method + "_version_ignored" if version_differs else method,
                        "DOI assigned using accession without version suffix; FASTA accession preserved."
                        if version_differs else "",
                    )
            else:
                report(acc, doi, candidate, status="not_in_dataset")
    result["source_doi"] = ["; ".join(values) for values in dois]
    return result, pd.DataFrame(audit, columns=columns)


## Optional filters — helper definitions

Define motif, sequence-length, and taxonomy filters. Enable the desired calls in the main pipeline or use them explicitly on a working dataframe. Check retained records before exporting a filtered dataset.

In [ ]:
# comment/uncomment as needed

# ==== CC MOTIFS ONLY ====
def filter_by_cc_motif(df):
    return df[df["motif"] != "NA"].copy()

# df_cc = filter_by_cc_motif(df)

# df_cc = df[df["motif"] != "NA"].reset_index(drop=True)
# print(f"{len(df_cc)} sequences with valid CC motifs")
# ========================

# ==== Curated accession list ====
def filter_by_accession_list(df, accession_set):
    return df[df["accession"].isin(accession_set)].copy()

# curated = set(pd.read_csv("curated_accessions.csv")["accession"])
# df = filter_by_accession_list(df, curated)
# ===============================

# ==== Filter by Taxonomy =====
SUPERGROUPS = {
    "protostomia": ["arthropoda", "mollusca", "annelida", "nematoda",
                    "platyhelminthes", "brachiopoda"],
    "deuterostomia": ["chordata", "echinodermata", "hemichordata"]
}

def expand_taxonomic_values(values):
    expanded = []
    for v in values:
        v_low = v.lower()
        if v_low in SUPERGROUPS:
            expanded.extend(SUPERGROUPS[v_low])
        else:
            expanded.append(v_low)
    return set(expanded)


def filter_by_taxonomy(df, field, values, exclude=False):
    """
    Filter dataframe by a taxonomy field.

    Parameters
    ----------
    field : str
        Taxonomy column (e.g. 'phylum', 'class')
    values : list[str]
        Values or supergroups to include/exclude
    exclude : bool
        If True, exclude matching values instead of include
    """
    field = field.lower()
    allowed = expand_taxonomic_values(values)

    df = df.copy()
    df[field] = df[field].fillna("").str.lower()

    mask = df[field].isin(allowed)

    if exclude:
        return df[~mask].reset_index(drop=True)
    else:
        return df[mask].reset_index(drop=True)
# ============================================


# # Final dataset for alignment
# df_align = df_cc.copy()

# log_stage("Alignment dataset defined", df_align)

# log_stage("Alignment-ready outputs")

# make_alignment_outputs(
#     df_align,
#     output_prefix="alignment_ready"
# )



## Output writers — helper definitions

Write FASTA records and metadata tables. Alignment headers link sequences to rows in the final metadata CSV; original-header output preserves source labels for tracing records back to the input.

In [ ]:
def write_uid_fasta(df, fasta_out):
    records = []

    for row in df.itertuples():
        species = str(row.species).strip() if pd.notna(row.species) else ""

        if species:
            species = re.sub(
                r"[^A-Za-z0-9]+",
                "_",
                species
            ).strip("_")
        else:
            species = "Unknown_species"

        uid = str(row.uid).strip()

        header = f"{species}_{uid}"

        records.append(
            SeqRecord(
                seq=Seq(str(row.sequence)),
                id=header,
                description=""
            )
        )

    SeqIO.write(records, fasta_out, "fasta")

    print(f"FASTA written: {fasta_out}")

def write_alignment_fasta(df, dir = OUTPUT_DIR):
    records = []
    path = Path(dir) / f"{DATASET_ID}_{RUN_DATE}_alignment_ready.fasta"
    for r in df.itertuples():
        rec = SeqRecord(
            seq=Seq(str(r.sequence)),
            id=r.alignment_header,
            description=""
        )
        records.append(rec)

    SeqIO.write(records, path, "fasta")


def write_metadata_csv(df, name):
    path = out(f"{name}.csv")
    df.to_csv(path, index=False)
    print(f"Wrote metadata CSV: {path}")


## Alignment preparation — helper definition

Build consistent headers using species and the selected identifier, with a local-ID fallback when an accession is missing. Repeated final labels receive copy suffixes. The final CSV records the exact `alignment_header` used in the FASTA.

In [ ]:
def make_alignment_outputs(df, output_prefix="alignment_ready", identifier="uid", header_suffix="", taxon_rank=None):
    """
    Create alignment-ready CSV and FASTA from a dataframe.
    Assumes df already contains only the sequences you want.
    """

    if identifier not in {"uid", "accession"}:
        raise ValueError("identifier must be uid or accession")

    if identifier not in df.columns:
        raise ValueError(f"{identifier!r} is unavailable. Set CREATE_UIDS=True to use UIDs.")
    suffix = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(header_suffix or "").strip()).strip("_")
    if header_suffix and not suffix:
        raise ValueError("header_suffix must contain a letter or number")

    if taxon_rank not in (None, "family", "order"):
        raise ValueError("taxon_rank must be None, 'family', or 'order'")
    if taxon_rank and taxon_rank not in df.columns:
        raise ValueError(f"{taxon_rank!r} column is unavailable; include it in TAXONOMIC_RANKS")

    # ---- sort deterministically ----
    df_sorted = df.sort_values(
        by=["species", identifier],
        na_position="last"
    ).reset_index(drop=True)

    # ---- alignment-safe header ----
    def make_alignment_header(row):
        # The species column already contains the full binomial name.
        species = row.get("species")
        species = str(species).strip() if pd.notna(species) else ""
        species = re.sub(r"[^A-Za-z0-9]+", "_", species).strip("_")
        species = species or "Unknown_species"

        sequence_id = row.get(identifier)
        sequence_id = str(sequence_id).strip() if pd.notna(sequence_id) else ""
        if sequence_id.lower() in {"", "na", "nan", "none"}:
            sequence_id = str(row.get("local_id", "")).strip()
        if not sequence_id:
            raise ValueError("Sequence has neither selected identifier nor local_id.")
        sequence_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", sequence_id).strip("_") or "NA"

        parts = [species, sequence_id]
        if suffix:
            parts.append(suffix)
        if taxon_rank:
            value = row.get(taxon_rank)
            value = str(value).strip() if pd.notna(value) else ""
            value = re.sub(r"[^A-Za-z0-9]+", "_", value).strip("_")
            parts.append(value or f"Unknown_{taxon_rank}")
        return "_".join(parts)

    df_sorted["alignment_header"] = df_sorted.apply(
        make_alignment_header, axis=1
    )

    # Keep every row; disambiguate repeated labels deterministically.
    used = set()
    headers = []
    for base in df_sorted["alignment_header"]:
        header = base
        number = 2
        while header in used:
            header = f"{base}_copy{number}"
            number += 1
        used.add(header)
        headers.append(header)
    df_sorted["alignment_header"] = headers

    # ---- CSV ----
    export_columns(df_sorted, FINAL_COLUMNS).to_csv(
        out(f"{output_prefix}.csv"),
        index=False
    )

    # ---- FASTA ----
    records = [
        SeqRecord(
            seq=Seq(str(row.sequence)),
            id=row.alignment_header,
            description=""
        )
        for row in df_sorted.itertuples()
    ]

    SeqIO.write(
        records,
        out(f"{output_prefix}.fasta"),
        "fasta"
    )

    print(f"Alignment-ready CSV and FASTA written: {output_prefix}")


    return df_sorted


## SignalP 6 trimming — optional helper definition

This notebook prepares input for an external SignalP run and reads its predictions; it does not run SignalP itself. Enable input export to obtain stable SignalP IDs, then set `SIGNALP_RESULTS` and the trimming option to apply predictions. With both export and trimming disabled, this step leaves sequences unchanged.

In [ ]:
def apply_signalp_predictions(df, results_path=None, trim=False):
    if not trim and not EXPORT_SIGNALP_INPUT:
        return df.copy()
    df = df.copy().reset_index(drop=True)
    df["signalp_id"] = [f"SP{i:06d}" for i in range(1, len(df) + 1)]
    input_fasta = out("signalp_input.fasta")
    SeqIO.write(
        [SeqRecord(Seq(str(r.sequence)), id=r.signalp_id, description="")
         for r in df.itertuples()], input_fasta, "fasta"
    )
    print(f"SignalP input FASTA: {input_fasta}")
    df["sequence_original"] = df["sequence"]
    df["signalp_prediction"] = ""
    df["signalp_cleavage_position"] = pd.NA
    df["signal_peptide_removed"] = False
    if not trim:
        return df
    if not results_path or not Path(results_path).is_file():
        raise FileNotFoundError(
            f"Run SignalP 6 on {input_fasta} and set SIGNALP_RESULTS to its "
            "prediction_results.txt before enabling REMOVE_SIGNAL_PEPTIDES."
        )
    with open(results_path, encoding="utf-8") as handle:
        lines = handle.readlines()
    header_line = next((line.lstrip("# ").strip() for line in lines
                        if line.lstrip("# ").startswith("ID\t")), None)
    if header_line is None:
        raise ValueError("SignalP results are missing the '# ID' tabular header")
    from io import StringIO
    predictions = pd.read_csv(
        StringIO(header_line + "\n" + "".join(line for line in lines
                   if line.strip() and not line.lstrip().startswith("#"))),
        sep="\t", dtype=str
    )
    predictions.columns = predictions.columns.str.strip()
    required = {"ID", "Prediction", "CS Position"}
    if not required.issubset(predictions.columns):
        raise ValueError(f"SignalP results must contain {sorted(required)} columns")
    if predictions["ID"].duplicated().any():
        raise ValueError("Duplicate IDs in SignalP predictions")
    predictions = predictions.set_index("ID")
    missing = set(df["signalp_id"]) - set(predictions.index)
    extra = set(predictions.index) - set(df["signalp_id"])
    if missing or extra:
        raise ValueError(f"SignalP IDs do not match this FASTA: {len(missing)} missing, {len(extra)} extra. "
                         "Run SignalP on the signalp_input.fasta produced by this run.")
    for idx, row in df.iterrows():
        prediction = predictions.loc[row["signalp_id"]]
        kind = str(prediction["Prediction"]).strip()
        df.at[idx, "signalp_prediction"] = kind
        if kind == "OTHER":
            continue
        # E.g. 'CS pos: 23-24. Pr: 0.98'; residue 23 is the final removed residue.
        match = re.search(r"(\d+)\s*-\s*(\d+)", str(prediction["CS Position"]))
        if not match:
            raise ValueError(f"Missing cleavage position for {row['signalp_id']}: {prediction['CS Position']!r}")
        cut = int(match.group(1))
        if not 0 < cut < len(row["sequence"]):
            raise ValueError(f"Invalid cleavage position {cut} for {row['signalp_id']}")
        df.at[idx, "signalp_cleavage_position"] = cut
        df.at[idx, "sequence"] = row["sequence"][cut:]
        df.at[idx, "signal_peptide_removed"] = True
    print(f"Removed predicted signal peptides from {df['signal_peptide_removed'].sum()} sequences")
    return df


## Preparation — run, then pause to edit the review CSV

This execution cell writes `metadata_review.csv`, attempts enabled lookups, and updates that sheet. Existing nonblank edits are imported first. Open the resulting CSV, complete unresolved entries, save it, and then run the main pipeline.

| Column | What to enter or preserve |
| --- | --- |
| `local_id` | Preserve exactly; joins the row to its FASTA record. |
| `original_header` | Input header for reference. |
| `species` | Full species name when known. |
| `gene` | Gene name or classification. |
| `accession` | Public accession when known; may remain blank for local sequences. |
| `identifier_source` | Origin of the identifier, when known. |
| `source_doi` | Paper DOI; matched from your source sheet where possible. |
| `mapping_status` | Use `manual` for a mapping you supplied and want preparation to preserve. |
| `missing_fields` | Automatically generated guide to unresolved fields. |

Recognized model identifiers can be searched online, but this is not a BLAST search. New protein accession assignments require an exact sequence match. Missing metadata is left for review and does not automatically exclude a sequence. Preparation also attempts DOI matching; `PREP_RUN_LOOKUPS` controls its remote requests. The separate DOI audit is written only when `WRITE_DOI_REPORT=True`.

In [ ]:

def local_model_id(header):
    # Recognized database model formats. Generic contig IDs are not searchable globally.
    match = re.search(r"(?:BGIBMGA\d+|HMEL\d+|MCINX\d+|Msex2[._]\d+|Px\d+\.\d+|OFAS\d+|AAEL\d+|AGAP\d+)", str(header))
    return match.group() if match else ""


def lookup_sequence_metadata(row, cache, online=True):
    """Use database records; assign a protein accession only on an exact sequence match."""
    import hashlib
    sequence = str(row["sequence"]).upper()
    accession = _doi_clean(row.get("accession", ""))
    source = _doi_clean(row.get("identifier_source", ""))
    model = local_model_id(row["original_header"])
    result = {"model_id": model, "mapping_status": "unresolved", "mapping_note": ""}
    # Never send local contigs or raw reads to Entrez as accessions.
    if source == "UniProt":
        result.update(mapping_status="uniprot_retained", mapping_note="UniProt ID retained; use header species or review sheet.")
        return result
    if not accession and not model:
        result["mapping_note"] = "No searchable database model ID; resolve species abbreviation using source paper."
        return result
    key = f"{accession or model}:{hashlib.sha256(sequence.encode()).hexdigest()}"
    if key in cache:
        return cache[key]
    if not online:
        result["mapping_note"] = "Online lookup disabled."
        return result
    try:
        if accession:
            db = "protein" if re.fullmatch(r"(?:(?:NP|XP|YP|WP|AP|ZP)_\d+|[A-Z]{3}\d+)(?:\.\d+)?", accession) else "nuccore"
            ids = [accession]
        else:
            db = "protein"
            query = f'"{model}"[All Fields]'
            species = _doi_clean(row.get("species", ""))
            if species:
                query += f' AND "{species}"[Organism]'
            with Entrez.esearch(db=db, term=query, retmax=PREP_MAX_CANDIDATES) as handle:
                search = Entrez.read(handle)
            time.sleep(0.35)
            if int(search["Count"]) > PREP_MAX_CANDIDATES:
                result.update(mapping_status="ambiguous", mapping_note="Too many model search results; review manually.")
                cache[key] = result
                return result
            ids = search["IdList"]
            if not ids:
                result["mapping_note"] = "No indexed NCBI protein cross-reference found."
                cache[key] = result
                return result
        with Entrez.efetch(db=db, id=",".join(ids), rettype="gp" if db == "protein" else "gb", retmode="text") as handle:
            records = list(SeqIO.parse(handle, "genbank"))
        time.sleep(0.35)
        candidates = []
        for record in records:
            organism = record.annotations.get("organism", "")
            if db == "protein":
                candidates.append((record.id, str(record.seq).upper(), organism))
            else:
                for feature in record.features:
                    if feature.type == "CDS":
                        for pid in feature.qualifiers.get("protein_id", []):
                            translations = feature.qualifiers.get("translation", [])
                            if translations:
                                candidates.append((pid, translations[0].upper(), organism))
        matches = list(dict.fromkeys((pid, organism) for pid, seq, organism in candidates if seq == sequence))
        if len(matches) == 1:
            pid, organism = matches[0]
            result.update(accession=pid, identifier_source="NCBI", species=organism,
                          mapping_status="exact_sequence_match", mapping_note=f"Exact protein sequence match to {pid}; local sequence preserved.")
        elif len(matches) > 1:
            organisms = {organism for _, organism in matches if organism}
            result.update(mapping_status="multiple_exact_matches", mapping_note="; ".join(pid for pid, _ in matches))
            if len(organisms) == 1:
                result["species"] = organisms.pop()
        else:
            result.update(mapping_status="sequence_match_not_confirmed", mapping_note="Candidate sequence differs, is partial, or has no annotated translation; accession not assigned.")
            # Existing NCBI accession directly identifies organism, even for nucleotide inputs.
            if accession and len(records) == 1:
                result["species"] = records[0].annotations.get("organism", "")
        cache[key] = result
    except Exception as error:
        result.update(mapping_status="lookup_failed", mapping_note=str(error))
        # Do not cache temporary network failures.
    return result


def prepare_metadata_review(fasta_path, online=True):
    """Run before main. No sequences are deduplicated, discarded, or replaced."""
    import json
    df = ingest_fasta(fasta_path)
    repeated = df[df["original_local_id"].duplicated(keep=False)]
    if len(repeated):
        print(f"Assigned unique internal IDs for {len(repeated)} records with repeated labels. See original_header in the review CSV.")
    df = apply_local_metadata(df, REVIEW_METADATA_CSV)
    # Fill from your existing DOI sheet BEFORE writing the editable worksheets.
    df, initial_doi_audit = annotate_source_dois(
        df, PAPER_ACCESSIONS_CSV, cache_file=DOI_LOOKUP_CACHE, allow_remote=False)
    write_doi_report(initial_doi_audit)
    print(f"DOI sheet: {PAPER_ACCESSIONS_CSV}; filled {(df["source_doi"] != "").sum()} records.")
    early_columns = ["local_id", "original_local_id", "original_header", "species", "gene", "source_doi", "accession", "identifier_source", "mapping_status", "model_id", "mapping_note"]
    Path(REVIEW_METADATA_CSV).parent.mkdir(parents=True, exist_ok=True)
    Path(SPECIES_REVIEW_CSV).parent.mkdir(parents=True, exist_ok=True)
    export_columns(df, REVIEW_COLUMNS).to_csv(REVIEW_METADATA_CSV, index=False)
    print(f"Editable CSV created: {Path(REVIEW_METADATA_CSV).resolve()}", flush=True)
    path = Path(PREP_LOOKUP_CACHE)
    cache = json.loads(path.read_text()) if path.exists() else {}
    for index, row in df.iterrows():
        if _doi_clean(row.get("mapping_status", "")) == "manual":
            continue
        metadata = lookup_sequence_metadata(row, cache, online=online)
        for column, value in metadata.items():
            if column not in df:
                df[column] = ""
            # Existing/manual species and accessions win unless a protein is sequence-verified.
            if column == "species" and _doi_clean(df.at[index, column]):
                continue
            if column == "accession" and metadata.get("mapping_status") != "exact_sequence_match":
                continue
            if value:
                df.at[index, column] = value
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(cache, indent=2))
    df, audit = annotate_source_dois(df, PAPER_ACCESSIONS_CSV, cache_file=DOI_LOOKUP_CACHE, allow_remote=online)
    write_doi_report(audit)
    def missing(row):
        needs = []
        for column in ["species", "source_doi", "accession"]:
            if not _doi_clean(row.get(column, "")):
                needs.append(column)
        return "; ".join(needs)
    df["missing_fields"] = df.apply(missing, axis=1)
    df["review_needed"] = df["missing_fields"].ne("")
    columns = ["local_id", "original_local_id", "original_header", "species", "gene", "accession", "identifier_source",
               "model_id", "mapping_status", "mapping_note", "source_doi", "missing_fields", "review_needed"]
    for column in columns:
        if column not in df:
            df[column] = ""
    export_columns(df, REVIEW_COLUMNS).to_csv(REVIEW_METADATA_CSV, index=False)
    species_missing = df[df["species"].map(_doi_clean).eq("")]
    print(f"Preparation retained all {len(df)} sequences. Missing species: {len(species_missing)}.")
    print(f"Full review sheet: {REVIEW_METADATA_CSV}")
    print("Blank metadata is allowed. Set mapping_status=manual for your own confirmed BLAST assignments.")
    return df


# Run this cell BEFORE the main pipeline. Edit the resulting CSVs, then run main.
prep_df = prepare_metadata_review(INPUT_FASTA, online=PREP_RUN_LOOKUPS)


## Pipeline Execution (main) — run after saving review edits

Read the FASTA and saved metadata review sheet, deduplicate sequences, resolve species, and apply enabled taxonomy/DOI annotations. Write the deduplicated original-header FASTA and alignment outputs. Review any interactive duplicate choices carefully because they determine which record's metadata is retained.

Filter examples in this cell are commented out until explicitly enabled. Run configuration and all helper definitions first; rerunning this execution cell rereads the input files.

In [ ]:
df = apply_local_metadata(ingest_fasta(INPUT_FASTA), LOCAL_METADATA_CSV)
log_stage("ingest", df)

# df = filter_by_min_length(df, MIN_LENGTH)
# log_stage("length_filter", df)


df = deduplicate(df, INTERACTIVE_DEDUP)
# New output file; the source FASTA is never overwritten.
original_records = [SeqRecord(Seq(str(row.sequence)), id=str(row.original_header).split()[0],
                              description=str(row.original_header)) for row in df.itertuples()]
SeqIO.write(original_records, out("deduplicated_original_headers.fasta"), "fasta")
log_stage("deduplicated_original_fasta", df, outputs=[out("deduplicated_original_headers.fasta")])
if PROMPT_FOR_UNIPROT_ID:
    df = prompt_for_missing_uniprot_ids(df)
log_stage("deduplicate", df)
if CREATE_UIDS:
    write_uid_fasta(df, out("deduplicated_UIDs.fasta"))

if PROMPT_FOR_SPECIES:
    df = resolve_unknown_species(df)
if ANNOTATE_TAXONOMY:
    df = annotate_taxonomy(df, TAXONOMIC_RANKS)
else:
    for rank in TAXONOMIC_RANKS:
        if rank not in df:
            df[rank] = ""
log_stage("taxonomy", df)

# Export the exact post-deduplication sequences for SignalP. To trim, run SignalP 6
# on this FASTA and set SIGNALP_RESULTS to its prediction_results.txt.
df = apply_signalp_predictions(df, results_path=SIGNALP_RESULTS, trim=REMOVE_SIGNAL_PEPTIDES)
log_stage("signalp", df)

# df["motif"] = df["sequence"].apply(extract_cc_motif)
# log_stage("cc_motif_extraction", df)

# Optional accession-to-paper DOI annotation (before final exports).
if ADD_SOURCE_DOIS:
    df, doi_review = annotate_source_dois(
        df, PAPER_ACCESSIONS_CSV, accession_column="accession",
        cache_file=DOI_LOOKUP_CACHE,
    )
    write_doi_report(doi_review)
    log_stage("source_dois", df)
    print(f"DOI annotation: {(df['source_doi'] != '').sum()} sequences with source DOIs")


# Create the final alignment-ready CSV and FASTA.
df_align = df.copy()
df_align = make_alignment_outputs(
    df_align,
    output_prefix="alignment_ready",
    identifier=ALIGNMENT_IDENTIFIER,
    header_suffix=HEADER_SUFFIX,
    taxon_rank=HEADER_TAXON_RANK
)


## InterProScan analysis — optional

Disabled by default. This section submits sequences to an external service and collects domain annotations. Before enabling `RUN_INTERPRO`, review its separate FASTA path, service settings, and email configuration. These optional service calls are separate from the core curation workflow.

### InterProScan helper definitions

Split records into batches and prepare results for merging. These functions support the optional execution cells below.

In [ ]:
def split_fasta_for_interpro(df, max_seqs=999):
    chunks = []
    for i in range(0, len(df), max_seqs):
        chunks.append(df.iloc[i:i+max_seqs])
    return chunks


# def write_fasta_from_df(df, fasta_path, header_col="uid"):
#     records = [
#         SeqRecord(
#             seq=Seq(str(row.sequence)),
#             id=row[header_col],
#             description=""
#         )
#         for row in df.itertuples()
#     ]
#     SeqIO.write(records, fasta_path, "fasta")

def write_fasta_from_df(df, fasta_path, header_col="uid"):
    records = []

    for row in df.itertuples(index=False):
        header = getattr(row, header_col)
        records.append(
            SeqRecord(
                Seq(row.sequence),
                id=str(header),
                description=""
            )
        )

    SeqIO.write(records, fasta_path, "fasta")
    print(f"Wrote FASTA: {fasta_path}")



def extract_domain_from_interpro(
    df,
    interpro_tsv,
    db="CDD",
    domain_name="DOMON_DOH"
):
    """
    Extract domain sequences based on InterProScan TSV.

    Returns a new dataframe with:
        - parent_uid
        - domain_sequence
        - domain_start
        - domain_end
        - domain_name
    """
    seq_map = dict(zip(df[ALIGNMENT_IDENTIFIER].astype(str), df["sequence"]))

    rows = []

    with open(interpro_tsv) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue

            parts = line.rstrip("\n").split("\t")
            if len(parts) < 8:
                continue

            uid, _, _, src_db, _, name, start, end = parts[:8]

            if src_db != db or name != domain_name:
                continue

            if uid not in seq_map:
                continue

            start = int(start)
            end = int(end)

            domain_seq = seq_map[uid][start-1:end]

            rows.append({
                "parent_uid": uid,
                "domain": domain_name,
                "domain_start": start,
                "domain_end": end,
                "sequence": domain_seq
            })

    return pd.DataFrame(rows)

def write_domain_fasta(domain_df, fasta_out):
    records = []
    for r in domain_df.itertuples():
        header = f"{r.parent_uid}|{r.domain}|{r.domain_start}-{r.domain_end}"
        records.append(
            SeqRecord(
                seq=Seq(str(r.sequence)),
                id=header,
                description=""
            )
        )
    SeqIO.write(records, fasta_out, "fasta")


### Run InterProScan — only when enabled

Check both execution cells below: one prepares batches and the other contains service-specific configuration and submission code. Replace example or old paths before running. Keep resulting annotations associated with the same FASTA identifiers.

In [ ]:
if RUN_INTERPRO:
    df_for_interpro = df_align   # or taxonomy-filtered df, otherwise filtered df

    chunks = split_fasta_for_interpro(df_for_interpro)

    for i, chunk in enumerate(chunks, start=1):
        fasta_path = out(f"interpro_chunk_{i}.fasta")
        write_fasta_from_df(chunk, fasta_path, header_col="alignment_header")


In [ ]:
import time
import requests
import sys
from pathlib import Path

# --- CONFIGURATION ---
INTERPROSCAN_API = "https://www.ebi.ac.uk/Tools/services/rest/iprscan5"
POLL_INTERVAL = 30  # seconds between checks
fasta_path = r"H:\My Drive\Katz-Lab_Otter-Data\Projects\Molecular_Neuroanatomy\analysis\TBH_domain_analysis\20260115\TBH-DBH_20260115_interpro_chunk_1.fasta"
out_filename = "interpro_scan_results_1"


# --- FUNCTIONS ---

def submit_job(fasta_path):
    """Submit FASTA file to InterProScan."""
    with open(fasta_path, "rb") as fasta:
        files = {"sequence": fasta}
        params = {
            "email": "your_email@domain.com",  # required field
            "goterms": "false",
            "pathways": "false",
            "outputFormat": "tsv",
        }
        print("Submitting FASTA file to InterProScan...")
        response = requests.post(f"{INTERPROSCAN_API}/run", files=files, data=params)
        if response.status_code != 200:
            print("Failed to submit job:", response.text)
            sys.exit(1)
        job_id = response.text.strip()
        print(f"Job submitted successfully. Job ID: {job_id}")
        return job_id


def check_status(job_id):
    """Check the status of an InterProScan job."""
    response = requests.get(f"{INTERPROSCAN_API}/status/{job_id}")
    if response.status_code != 200:
        print("Failed to check job status:", response.text)
        sys.exit(1)
    return response.text.strip()


def download_result(job_id, output_file):
    """Download the TSV result of the job."""
    print("Downloading TSV results...")
    url = f"{INTERPROSCAN_API}/result/{job_id}/tsv"
    response = requests.get(url)
    if response.status_code != 200:
        print("Failed to download results:", response.text)
        sys.exit(1)
    with open(output_file, "wb") as f:
        f.write(response.content)
    print(f"Results saved to {output_file}")


def main():

    job_id = submit_job(fasta_path)

    # Poll until the job is finished
    while True:
        status = check_status(job_id)
        print(f"Job status: {status}")
        if status in ("FINISHED", "FAILURE", "NOT_FOUND"):
            break
        time.sleep(POLL_INTERVAL)

    if status == "FINISHED":
        output_path = out("interpro_scan_results_1.tsv")
        download_result(job_id, output_path)
    else:
        print("Job failed or not found.")


if RUN_INTERPRO and __name__ == "__main__":
    main()


### Merge InterProScan results — optional

Combine returned TSV results into one CSV when `RUN_INTERPRO` is enabled. Run after the analysis has produced its result files.

In [ ]:
import pandas as pd
from pathlib import Path

def merge_interpro_tsvs_to_csv(interpro_dir, csv_out):
    interpro_dir = Path(interpro_dir)
    tsv_files = sorted(interpro_dir.glob("*.tsv"))

    rows = []
    skipped = []

    for f in tsv_files:
        with open(f, "r", errors="replace") as fh:
            lines = [
                line.rstrip("\n")
                for line in fh
                if line.strip() and not line.startswith("#")
            ]

        if not lines:
            skipped.append(f.name)
            continue

        for line in lines:
            rows.append(line.split("\t"))

    if not rows:
        raise ValueError("No InterPro data rows found in any TSV file.")

    df = pd.DataFrame(rows)

    df.to_csv(csv_out, index=False, header=False)
    print(f"Merged CSV written: {csv_out}")

    if skipped:
        print(f"Skipped {len(skipped)} TSVs with no data:")
        for s in skipped:
            print(f"  - {s}")

    return df


if RUN_INTERPRO:
    merge_interpro_tsvs_to_csv(
        OUTPUT_DIR,
        out("interpro_merged.csv")
    )


### Dibasic cleavage and peptide chunks — optional

When `RUN_PEPTIDE_ANALYSIS=True`, split sequences at `KR` or `RR` motifs and export candidate chunks. This is a motif-based heuristic, not a validated prediction of mature peptides. Review its assumptions and limits before interpreting the output.

In [ ]:
import re

DIBASIC_RE = re.compile(r"(KR|RR)")

def extract_peptide_chunks(
    seq,
    uid,
    max_chunks=6,
):
    """
    Returns a list of peptide dicts.
    """

    chunks = []
    cleavage_sites = [m.start() for m in DIBASIC_RE.finditer(seq)]

    # Split positions
    cut_positions = [0] + [i+2 for i in cleavage_sites] + [len(seq)]

    for i in range(len(cut_positions) - 1):
        start = cut_positions[i]
        end = cut_positions[i+1]
        peptide = seq[start:end]

        if not peptide:
            continue

        chunks.append({
            ALIGNMENT_IDENTIFIER: uid,
            "peptide_index": i + 1,
            "start": start + 1,          # 1-based
            "end": end,
            "length": len(peptide),
            "cysteines": peptide.count("C"),
            "sequence": peptide,
            "type": "mature_peptide"
        })

    if len(chunks) > max_chunks:
        raise ValueError(f"{uid}: more than {max_chunks} peptide chunks")

    return chunks, len(cleavage_sites)

if RUN_PEPTIDE_ANALYSIS:
    peptide_rows = []
    cleavage_summary = []

    for row in df.itertuples():
        try:
            peptides, n_cuts = extract_peptide_chunks(
                row.sequence,
                getattr(row, ALIGNMENT_IDENTIFIER)
            )
            peptide_rows.extend(peptides)
            cleavage_summary.append({
                ALIGNMENT_IDENTIFIER: getattr(row, ALIGNMENT_IDENTIFIER),
                "cleavage_sites": n_cuts
            })
        except ValueError as e:
            print(f"Skipping {getattr(row, ALIGNMENT_IDENTIFIER)}: {e}")

    df_peptides = pd.DataFrame(peptide_rows)
    df_peptides.to_csv(out("peptide_chunks.csv"), index=False)

    df_cleavage = pd.DataFrame(cleavage_summary)
    df = df.merge(df_cleavage, on=ALIGNMENT_IDENTIFIER, how="left")


## CC motif extraction — helper definition

Extract a region around the first adjacent `CC` motif. This is a specialized optional transformation; defining the function alone does not apply it to the input sequences.

In [ ]:
def extract_cc_motif(seq, upstream=5, downstream=35):
    for m in re.finditer("CC", seq):
        start = max(0, m.start() - upstream)
        end = min(len(seq), m.end() + downstream)
        window = seq[start:end]

        motif = []
        i = 0
        while i < len(window):
            if window[i] == "C":
                motif.append("C")
                i += 1
            else:
                j = i
                while j < len(window) and window[j] != "C":
                    j += 1
                motif.append(f"{j-i}X")
                i = j

        motif = "".join(motif)
        motif = re.sub(r"^\d+X|\d+X$", "", motif)

        if motif.count("C") >= 3:
            return motif

    return "NA"

## Additional filtering — optional examples

Use these examples to create a filtered working dataset. If you change the retained records, regenerate both the alignment FASTA and its metadata CSV from that same dataset.

Insulin-like peptide example: Retain full sequences containing a CC motif for further inspection. This broad screen can include unrelated proteins and miss incomplete sequences.
TBH example: Retain full sequences annotated with the DOMON domain in an InterProScan result file. Check that the annotation database, domain label, and sequence IDs match your file.
Filtering retains full sequences. Extracting only a domain or a region around a motif is a separate step and should produce a separately named dataset.

In [ ]:
# OPTIONAL FILTERING
# Run after the main pipeline.
#
# Examples from previous projects
# - Insulin-like peptides: retain sequences containing adjacent cysteines (CC).
# - TBH: retain full-length sequences with a DOMON domain annotation.
#
# These features help select candidates. They do not establish gene identity.
# Leave both switches False to skip this cell.

FILTER_FOR_CC = False
FILTER_FOR_DOMON = False

# For DOMON filtering, supply an InterProScan TSV produced from this dataset.
# Its sequence IDs must match the alignment_header values in df_align.
INTERPRO_TSV = None  # Example: Path("interpro_results.tsv")

# These values must match the annotation labels in your TSV.
DOMON_DATABASE = "CDD"
DOMON_LABEL = "DOMON_DOH"

# Choose a descriptive output name.
FILTERED_OUTPUT_PREFIX = "filtered_alignment_ready"


if FILTER_FOR_CC and FILTER_FOR_DOMON:
    raise ValueError("Choose one example filter at a time.")

if FILTER_FOR_CC or FILTER_FOR_DOMON:
    # Preserve the main dataset and filter a separate copy.
    filtered = df_align.copy()
    starting_count = len(filtered)

    if FILTER_FOR_CC:
        # Retain the full sequence whenever it contains a CC motif.
        # This does not extract the mature insulin-like peptide.
        keep = filtered["sequence"].fillna("").str.contains(
            "CC", case=False, regex=False
        )
        filtered = filtered.loc[keep].copy()
        filter_description = "adjacent cysteines (CC)"

    elif FILTER_FOR_DOMON:
        if INTERPRO_TSV is None:
            raise ValueError("Set INTERPRO_TSV before filtering for DOMON.")

        # Collect IDs annotated with the requested domain.
        matching_ids = set()

        with open(INTERPRO_TSV) as handle:
            for line in handle:
                if not line.strip() or line.startswith("#"):
                    continue

                fields = line.rstrip("\n").split("\t")
                if len(fields) < 8:
                    continue

                sequence_id = fields[0]
                database = fields[3]
                domain_label = fields[5]

                if (
                    database == DOMON_DATABASE
                    and domain_label == DOMON_LABEL
                ):
                    matching_ids.add(sequence_id)

        keep = filtered["alignment_header"].isin(matching_ids)
        filtered = filtered.loc[keep].copy()
        filter_description = f"{DOMON_DATABASE}: {DOMON_LABEL}"

    print(
        f"Retained {len(filtered)} of {starting_count} sequences "
        f"using {filter_description}."
    )

    if filtered.empty:
        raise ValueError(
            "No sequences matched. Check the filter, annotation labels, "
            "and sequence IDs before exporting."
        )

    df_filtered_align = make_alignment_outputs(
        filtered,
        output_prefix=FILTERED_OUTPUT_PREFIX,
        identifier=ALIGNMENT_IDENTIFIER,
        header_suffix=HEADER_SUFFIX,
        taxon_rank=HEADER_TAXON_RANK,
    )

else:
    print("Optional filtering skipped.")